In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("API key is set.")

API key is set.


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# **RAG IMPLEMENTATION WITH PDF**

#### **STEP 1: Extracting Text from PDFs**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Team 2 - Final Report.pdf"

loader = PyPDFLoader(pdf_path) # It will load that pdf for us. And bydefault it will add the metadata as well.

docs = loader.load()

docs

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-06T23:03:03-07:00', 'author': 'Darpan Jiyani', 'moddate': '2026-04-06T23:03:03-07:00', 'source': './Docs/Team 2 - Final Report.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \nSimulation of Domain Expert in LLM models \n \n \n \n \n \n \nA Project Report \nPresented to  \nThe Faculty of the Department of Applied Data Science \nSan Jose State University \nIn Partial Fulfillment \nOf the Requirements for the Degree \nMaster of Science in Applied Data Intelligence \n \n \n \n \n \n \n \n \n \n \n \n \n \nBy \n \nSaqib Chowdhury \nDarpankumar Jiyani \nPrathyusha Pingili \nManjot Singh \nSai Prasad Thalluri \n \n \n \nDecember 5, 2025'),
 Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-04-06T23:03:03-07:00', 'author': 'Darpan Jiyan

Here It creats so many document because we have a huge pdf.

### **Creating our own Metadata for PDF chunks**

In [6]:
for i in docs:    #List have capability to make in place changes. So we can add the metadata to each document in the list.

    i.metadata = {"source": "team 2 Final Report.pdf",
                  "People": "All Team Members"}

In [7]:
docs[0].metadata

{'source': 'team 2 Final Report.pdf', 'People': 'All Team Members'}

Above way we can overwrite our metadata

#### **STEP 2: Splitting the Document into CHUNKS**

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'team 2 Final Report.pdf', 'People': 'All Team Members'}, page_content='1 \n \n \nSimulation of Domain Expert in LLM models \n \n \n \n \n \n \nA Project Report \nPresented to  \nThe Faculty of the Department of Applied Data Science \nSan Jose State University \nIn Partial Fulfillment \nOf the Requirements for the Degree \nMaster of Science in Applied Data Intelligence \n \n \n \n \n \n \n \n \n \n \n \n \n \nBy \n \nSaqib Chowdhury \nDarpankumar Jiyani \nPrathyusha Pingili \nManjot Singh \nSai Prasad Thalluri \n \n \n \nDecember 5, 2025'),
 Document(metadata={'source': 'team 2 Final Report.pdf', 'People': 'All Team Members'}, page_content='2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nCopyright © 2025 \nSaqib Chowdhury, Darpankumar Jiyani, Prathyusha Pingili, Manjot Singh, Sai Prasad Thalluri \nALL RIGHTS RESERVED'),
 Document(metadata={'source': 'team 2 Final Report.pdf', 'Peop

In [9]:
len(chunks)

221

In [10]:
chunks[0].metadata

{'source': 'team 2 Final Report.pdf', 'People': 'All Team Members'}

We didn't create above metadata. It created for us. if we using any pdf, document then langchain pypdf loader will automatically create the metadata for us when we will be creating chunks.

We have 221 chunks. That means we'll be having 221 vectors embedding vectors for it.

#### **STEP 3: Creating Embeddings for the Chunks**

In [15]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

#### **STEP 4: Create and Store Embeddings in Local Vector Store**

In [12]:
from langchain_community.vectorstores import Chroma

vectorestore = Chroma.from_documents(
    documents = chunks, 
    embedding = embedding_model,
    persist_directory="./DB/")

#### Now if we will do the restart then we can perform our similarity search on our Local DB directly without even performing all the above things, why?

#### Because we already have our vectors store in DB. It's not gone. we can now just query it.

#### Now how can we create that connection?

#### **RE-USE the DB**

In [14]:
vectorestore_persist = Chroma(
    persist_directory="./DB/", 
    embedding_function=embedding_model)

C:\Users\darpa\AppData\Local\Temp\ipykernel_30828\2201963593.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorestore_persist = Chroma(


In [16]:
vectorestore_persist.similarity_search("What is the main topic of the report?", k=3)

[Document(metadata={'People': 'All Team Members', 'source': 'team 2 Final Report.pdf'}, page_content='"Mixture of Endpoints". Overall, this report represents a complete end-to-end methodology to \nbuild industry-leading LLM systems from the initial data collection stage to when they can be \nutilized by users. This report has shown that open-source models can achieve expert level'),
 Document(metadata={'People': 'All Team Members', 'source': 'team 2 Final Report.pdf'}, page_content='6 \n \nTABLE OF CONTENTS \n \nChapter 1 Introduction                                                                                        \n1.1   Project Background and Execute Summary                                                       \n1.2   Project Requirements                                                                                        \n1.3   Project Deliverables                                                                                           \n1.4   Technology and Solution Surv

This is our efficient way to persist our RAG.

Is there any way that we can store more and more data. For larger application??



Because we want to make our chatbot robust enought to handle all the request.

We can add more features and documents in the existing RAG.